In [0]:
import json
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType


COURSE_ID = 'py_eng'


# 1. Bronze table nundi video_id and channel_name lookup techukundham
# Ee lookup dictionary dwara manam fast ga channel name ni match cheyochu
bronze_data = spark.sql(f"SELECT video_id, channel_name FROM courseify.default.course_bronze WHERE course_id = '{COURSE_ID}'").collect()
video_to_channel = {row['video_id']: row['channel_name'] for row in bronze_data}

# 2. JSON update cheyadaniki oka UDF (User Defined Function)
def add_channel_to_json(json_str):
    if not json_str:
        return json_str
    try:
        # JSON string ni list of dictionaries ga marchali
        video_list = json.loads(json_str)
        
        # Prathi video object ki channel name ni lookup nundi assign chesthunnam
        for video in video_list:
            v_id = video.get('video_id')
            # Bronze table lookup lo unte adi add chestham, lekapothe 'Unknown'
            video['channel'] = video_to_channel.get(v_id, "Unknown")
            
        # Malli JSON string ga convert chesthunnam
        return json.dumps(video_list)
    except Exception as e:
        return json_str

# UDF ni register chesthunnam
update_json_udf = udf(add_channel_to_json, StringType())

# 3. Silver table ni load chesi column ni update chedhdam
silver_df = spark.sql(f"SELECT * FROM courseify.default.course_silver WHERE course_id = '{COURSE_ID}'")

# sub_videos_json column ki update apply chesthunnam
updated_silver_df = silver_df.withColumn("sub_videos_json", update_json_udf(col("sub_videos_json")))

# 4. Updated data ni temporary view ga create chesi MERGE chesthunnam
updated_silver_df.createOrReplaceTempView("updated_silver_data")

spark.sql(f"""select * from updated_silver_data""").display()

spark.sql("""
    MERGE INTO courseify.default.course_silver target
    USING updated_silver_data source
    ON target.course_id = source.course_id AND target.main_topic_id = source.main_topic_id
    WHEN MATCHED THEN
      UPDATE SET
        target.sub_videos_json = source.sub_videos_json
""")

print("✅ Silver table lo unna existing JSON data ki channel names successfully update ayyay!")